# Img2GPS — Walkthrough

CIS 5190 final project, Track A. We predict GPS coordinates from a single image taken on Penn's campus (test rectangle: 33rd & Walnut → 34th & Spruce). The official metric is the average Haversine distance in meters.

This notebook is the human-readable companion to the scripts:

- `Img2GPS/extract_exif.py` builds `metadata.csv`
- `Img2GPS/preprocess.py` provides `prepare_data` / `load_raw`
- `Img2GPS/model.py` defines the ResNet-18 regressor with target-normalization buffers
- `Img2GPS/train.py` runs the training loop
- `Img2GPS/eval_project_a.py` is the course-style evaluator


# Colab bootstrap
clone the repo and install dependencies on first run. Safe to re-run locally; it's a no-op when the repo already exists.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/SheilaBkny/cis_5190_project.git"
REPO_BRANCH = "iter1"
COLAB_REPO_DIR = "/content/cis_5190_project"

IN_COLAB = "google.colab" in sys.modules

def _git(*args):
    subprocess.run(["git", "-C", COLAB_REPO_DIR, *args], check=True)

if IN_COLAB:
    if not os.path.exists(os.path.join(COLAB_REPO_DIR, "Img2GPS")):
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, COLAB_REPO_DIR],
            check=True,
        )
    else:
        # Repo dir already exists from an earlier session — pull the
        # latest commit on iter1 so new data / notebook fixes show up
        # without needing to delete /content/cis_5190_project manually.
        _git("fetch", "origin", REPO_BRANCH)
        _git("checkout", REPO_BRANCH)
        _git("reset", "--hard", f"origin/{REPO_BRANCH}")
    os.chdir(COLAB_REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
    head = subprocess.check_output(["git", "-C", COLAB_REPO_DIR, "log", "-1", "--oneline"]).decode().strip()
    print(f"on commit: {head}")

print("cwd:", os.getcwd())

In [ ]:
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / "Img2GPS").exists():
    REPO_ROOT = REPO_ROOT.parent
PROJECT_DIR = REPO_ROOT / "Img2GPS"
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(REPO_ROOT)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("count :", torch.cuda.device_count())
    print("mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("torch :", torch.__version__)

from preprocess import load_raw, prepare_data  # noqa: E402
from model import Model  # noqa: E402
from train import haversine_meters, location_grouped_split  # noqa: E402

REPO_ROOT, PROJECT_DIR

## 1. Data summary

We collected 89 photos around Penn's campus and extracted GPS coordinates from EXIF + Apple location xattrs. The location-grouped split makes sure photos that share an exact GPS coordinate (about 8 photos per spot) live entirely on one side of the train/val split, which avoids leakage.

In [ ]:
df = pd.read_csv(PROJECT_DIR / "metadata.csv")
print(f"rows: {len(df)}")
print(f"unique locations: {df[['latitude','longitude']].drop_duplicates().shape[0]}")
df.describe(percentiles=[0.05, 0.5, 0.95]).round(6)

In [ ]:
def haversine(a, b):
    R = 6_371_000.0
    lat1, lon1 = map(math.radians, a)
    lat2, lon2 = map(math.radians, b)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

mean_lat = df['latitude'].mean()
mean_lon = df['longitude'].mean()
lat_span = haversine((df['latitude'].min(), mean_lon), (df['latitude'].max(), mean_lon))
lon_span = haversine((mean_lat, df['longitude'].min()), (mean_lat, df['longitude'].max()))
print(f"bounding box: {lat_span:.1f} m (NS) x {lon_span:.1f} m (EW)")
print(f"training mean: ({mean_lat:.6f}, {mean_lon:.6f})")

constant_dist = [haversine((mean_lat, mean_lon), p) for p in df[['latitude','longitude']].values]
print(f"constant-mean baseline Haversine: mean={np.mean(constant_dist):.2f}m  p50={np.median(constant_dist):.2f}m  max={np.max(constant_dist):.2f}m")

In [ ]:
# PHONE-ONLY by default. The leaderboard scored 82.26 m for phone-only
# (n=89) vs 101.93 m for the combined+aug+finetune model — Mapillary's
# car-mounted dashcam frames don't bridge into the phone-on-walkway
# test domain even with heavy domain-randomization aug. Set
# USE_MAPILLARY=True only to re-run that experiment.
USE_MAPILLARY = True

main_csv = PROJECT_DIR / "metadata.csv"
extra_csv = PROJECT_DIR / "metadata_mapillary.csv"
combined_csv = PROJECT_DIR / "metadata_combined.csv"

if USE_MAPILLARY and extra_csv.exists():
    frames = [
        pd.read_csv(main_csv).assign(source="phone"),
        pd.read_csv(extra_csv).assign(source="mapillary"),
    ]
    combined = pd.concat(frames, ignore_index=True)
    combined.drop(columns=["source"]).to_csv(combined_csv, index=False)
    TRAIN_CSV = str(combined_csv)
    print(f"main : {len(frames[0])} rows  ({main_csv.name})")
    print(f"extra: {len(frames[1])} rows  ({extra_csv.name})")
    print(f"combined -> {combined_csv.name}: {len(combined)} rows")
    print("breakdown:", combined.groupby("source").size().to_dict())
else:
    TRAIN_CSV = str(main_csv)
    print(f"phone-only training: {len(pd.read_csv(main_csv))} rows  ({main_csv.name})")
TRAIN_CSV

## 2. Train

Trains on whatever `TRAIN_CSV` was set to in the previous cell — phone-only by default (`metadata.csv`, 89 rows). The script saves the best-by-phone-val checkpoint to `Img2GPS/model.pt`. Skip this cell if you want to evaluate the existing checkpoint.

In [ ]:
# --phone-csv lets train.py route Mapillary samples to the heavier
# domain-randomization augmentation pipeline (harmless when TRAIN_CSV
# is the phone CSV — every sample is then "phone").
# --finetune-epochs only does work when the main loop trained on a
# mixed-domain set, so it scales with USE_MAPILLARY.
finetune_epochs = 4 if USE_MAPILLARY else 0
print(f"USE_MAPILLARY={USE_MAPILLARY}  finetune_epochs={finetune_epochs}")
!python Img2GPS/train.py --csv "{TRAIN_CSV}" \
    --phone-csv "{main_csv}" \
    --epochs 12 --lr 1e-3 --finetune-epochs {finetune_epochs}

## 2.5 Save the trained model to your machine

`Img2GPS/model.pt` only exists inside the Colab VM after training — it is *not* automatically synced back to your laptop or to the `submission` branch. The next cell:

1. Prints the file's size and a short MD5 fingerprint so you can confirm it's the run you just produced.
2. If running in Colab, triggers a browser download.

After the download completes, place the file at `Img2GPS/model.pt` in your local clone and run `Img2GPS/scripts/promote_to_submission.sh` to copy it onto the `submission` branch and push — that's the version the leaderboard pulls.

In [ ]:
import hashlib, os, sys

MODEL_PATH = str(PROJECT_DIR / "model.pt")
size_mb = os.path.getsize(MODEL_PATH) / 1e6
md5 = hashlib.md5(open(MODEL_PATH, "rb").read()).hexdigest()[:12]
print(f"model.pt   size: {size_mb:.1f} MB   md5: {md5}")

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(MODEL_PATH)
    print("Downloaded. Move ~/Downloads/model.pt -> <repo>/Img2GPS/model.pt locally,")
    print("then: bash Img2GPS/scripts/promote_to_submission.sh")
else:
    print("Not in Colab — model.pt is already on your local filesystem at the path above.")

## 3. Evaluate the saved checkpoint

We load `model.pt` (the best-by-Haversine snapshot) and report the official metrics on:

1. The full collected dataset.
2. The held-out validation split (location-grouped, the honest signal).
3. The course-provided reference set in `Img2GPS/reference/`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Model(weights_path=str(PROJECT_DIR / 'model.pt')).to(device).eval()
print(f"y_mean: {model.y_mean.tolist()}\ny_std:  {model.y_std.tolist()}")

In [ ]:
def evaluate(csv_path):
    X, y = prepare_data(str(csv_path))
    with torch.no_grad():
        preds = model(X.to(device)).cpu()
    distances = haversine_meters(preds, y).numpy()
    return preds, y, distances

preds_full, y_full, dists_full = evaluate(PROJECT_DIR / 'metadata.csv')
print(f"FULL set (n={len(y_full)}):")
print(f"  mean={dists_full.mean():.2f}m  p50={np.median(dists_full):.2f}m  p90={np.quantile(dists_full,0.9):.2f}m  max={dists_full.max():.2f}m")

In [ ]:
_, y_all = load_raw(str(PROJECT_DIR / 'metadata.csv'))
train_idx, val_idx = location_grouped_split(y_all, val_fraction=0.2, seed=42)
X_full, _ = prepare_data(str(PROJECT_DIR / 'metadata.csv'))
with torch.no_grad():
    preds_val = model(X_full[val_idx].to(device)).cpu()
dists_val = haversine_meters(preds_val, y_all[val_idx]).numpy()
print(f"VAL set (held-out, n={len(val_idx)}):")
print(f"  mean={dists_val.mean():.2f}m  p50={np.median(dists_val):.2f}m  p90={np.quantile(dists_val,0.9):.2f}m  max={dists_val.max():.2f}m")

In [ ]:
ref_csv = PROJECT_DIR / 'reference' / 'metadata.csv'
preds_ref, y_ref, dists_ref = evaluate(ref_csv)
print(f"REFERENCE set (n={len(y_ref)}):")
for (lat, lon), (plat, plon), d in zip(y_ref.tolist(), preds_ref.tolist(), dists_ref):
    print(f"  truth=({lat:.6f},{lon:.6f})  pred=({plat:.6f},{plon:.6f})  haversine={d:.2f}m")
print(f"reference mean Haversine: {dists_ref.mean():.2f}m")